# Day 11: Build an NFL star schema

**Phase 1: Foundations & Infrastructure**

## Objective
Build an NFL star schema

## Task
- Diseñamos una arquitectura de **Esquema de Estrella (Star Schema)** específica para la NFL, separando las jugadas (Hechos) de su contexto (Dimensiones).
- Trazamos un diagrama de entidad-relación (ERD) usando Mermaid para visualizar cómo las llaves primarias (`game_id`, `drive_id`, `player_id`) conectan todo el modelo.
- Implementamos un pipeline ETL en Pandas donde:
  - *Extraímos* los datos usando nuestra función unificada `load_nfl`.
  - *Transformamos* la data plana construyendo dimensiones como `dim_games`, `dim_teams`, `dim_players` (limpiando el roster) y `dim_drives` (creando llaves compuestas únicas como `game_id + drive_number`).
  - *Cargamos* los resultados exportándolos a formato ultra-optimizado `.parquet` (usando `pyarrow`) en `data/processed/nfl/`.
- **Bugfixes Críticos Resueltos:** 
  - Aplicamos un bypass de seguridad SSL en `loaders.py` para permitir la descarga ininterrumpida de nflfastR en macOS.
  - Actualizamos la llamada de la librería de `import_rosters` a `import_seasonal_rosters`.
  - Corregimos el schema del roster para utilizar correctamente `player_id` en lugar del obsoleto `gsis_id`.

**Deliverable:** ERD diagram + transformation script + processed data in data/processed/nfl/.

---

### Instructions:
*Treat this as your course workbook. Write your code, notes, or execution steps below.*

Remember: 
- Document your decisions.
- Use clear variable names.
- If today's task involves creating a script in `src/`, a dashboard in `apps/`, or documentation in `docs/`, you can use this notebook to test your logic or simply write your reflections and proofs of execution (e.g. running terminal commands with `!`).



### NFL Star Schema - Entity Relationship Diagram

```mermaid
erDiagram
    fact_plays {
        string play_id PK
        string game_id FK
        string drive_id FK
        string passer_player_id FK
        string rusher_player_id FK
        string posteam FK
        string defteam FK
        int down
        int ydstogo
        int yardline_100
        float epa
        float yards_gained
    }
    
    dim_games {
        string game_id PK
        int season
        int week
        string home_team
        string away_team
    }
    
    dim_drives {
        string drive_id PK
        string game_id FK
        string posteam
        string drive_result
        int drive_play_count
    }
    
    dim_players {
        string player_id PK
        string player_name
        string position
        string college
    }
    
    dim_teams {
        string team_id PK
        string team_name
        string team_abbr
    }

    dim_games ||--o{ fact_plays : "contains"
    dim_drives ||--o{ fact_plays : "groups"
    dim_players ||--o{ fact_plays : "executes"
    dim_teams ||--o{ fact_plays : "participates_in"
```


In [1]:
import sys
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 1. Path configuration and imports
sys.path.append(os.path.abspath('../../')) 
from src.data import loaders

PROCESSED_DIR = "../../data/processed/nfl/"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("--- 🏈 Starting NFL ETL Pipeline (Star Schema) ---")

# --- EXTRACT ---
print("\n1. Extracting data...")
# Using your exact loader function for the 2023 season
SEASON = [2023]
pbp_df = loaders.load_nfl(years=SEASON, data_type="pbp")
roster_df = loaders.load_nfl(years=SEASON, data_type="roster")

# --- TRANSFORM ---
print("\n2. Transforming dimensions and facts...")

# A. Games Dimension (dim_games)
dim_games = pbp_df[['game_id', 'season', 'week', 'home_team', 'away_team']].dropna(subset=['game_id']).drop_duplicates(subset=['game_id'])

# B. Teams Dimension (dim_teams)
# We extract unique teams from the home_team column. In nflfastR, abbreviations act as IDs.
dim_teams = pd.DataFrame({'team_id': pbp_df['home_team'].dropna().unique()})
dim_teams['team_abbr'] = dim_teams['team_id'] 
dim_teams['team_name'] = dim_teams['team_abbr'] # Can be enriched later with full names

# C. Players Dimension (dim_players)
# nflfastR roster data uses 'player_id' which maps to 'passer_player_id' etc. in PBP
if roster_df is not None:
    dim_players = roster_df[['player_id', 'player_name', 'position', 'college']].copy()
    dim_players.dropna(subset=['player_id'], inplace=True)
    dim_players.drop_duplicates(subset=['player_id'], inplace=True)
else:
    print("Warning: Roster data missing. dim_players will be empty.")
    dim_players = pd.DataFrame(columns=['player_id', 'player_name', 'position', 'college'])

# D. Drives Dimension (dim_drives)
# Create a globally unique drive_id since 'drive' resets every game
pbp_df['drive_id'] = pbp_df['game_id'] + "_" + pbp_df['drive'].astype(str)
dim_drives = pbp_df[['drive_id', 'game_id', 'posteam', 'fixed_drive_result', 'drive_play_count']].dropna(subset=['drive_id']).drop_duplicates(subset=['drive_id'])

# E. Fact Table (fact_plays)
# Create a globally unique play_id
pbp_df['global_play_id'] = pbp_df['game_id'] + "_" + pbp_df['play_id'].astype(str)

fact_columns = [
    'global_play_id', 'game_id', 'drive_id', 
    'passer_player_id', 'rusher_player_id', 'receiver_player_id',
    'posteam', 'defteam', 'down', 'ydstogo', 'yardline_100', 
    'epa', 'yards_gained', 'play_type'
]
existing_cols = [col for col in fact_columns if col in pbp_df.columns]
fact_plays = pbp_df[existing_cols].copy()

# Rename global_play_id to play_id to match our ERD
fact_plays.rename(columns={'global_play_id': 'play_id'}, inplace=True)

# --- LOAD ---
print("\n3. Loading to Parquet format...")
dim_games.to_parquet(f"{PROCESSED_DIR}dim_games.parquet", index=False)
dim_teams.to_parquet(f"{PROCESSED_DIR}dim_teams.parquet", index=False)
dim_players.to_parquet(f"{PROCESSED_DIR}dim_players.parquet", index=False)
dim_drives.to_parquet(f"{PROCESSED_DIR}dim_drives.parquet", index=False)
fact_plays.to_parquet(f"{PROCESSED_DIR}fact_plays.parquet", index=False)

print(f"✅ Pipeline complete! NFL Star Schema ready at: {PROCESSED_DIR}")


2026-05-17 13:40:56 - INFO - Local cache found. Loading Parquet data from: c:\Users\david\Desktop\David\Documentos\David\GitHub\100-days-Data-Sports-Challenge\data\raw/nfl/nfl_pbp_2023.parquet


--- 🏈 Starting NFL ETL Pipeline (Star Schema) ---

1. Extracting data...


2026-05-17 13:40:56 - INFO - Downloading NFL roster data for years: [2023]
2026-05-17 13:40:57 - INFO - Data successfully saved to c:\Users\david\Desktop\David\Documentos\David\GitHub\100-days-Data-Sports-Challenge\data\raw/nfl/nfl_roster_2023.parquet



2. Transforming dimensions and facts...

3. Loading to Parquet format...
✅ Pipeline complete! NFL Star Schema ready at: ../../data/processed/nfl/
